# 为什么 Quantizer 不需要继承 `nn.Module`

一句话结论：**继承 `nn.Module` 会让 `self.x = y` 这行普通赋值变成一个「记账动作」。**

这个「账本」对神经网络很有用（`.to("cuda")`、`.parameters()` 全靠它），
但对 `AwqQuantizer` 完全没用 —— 它只是个改权重的脚本，不参与前向。

下面逐个跑出来看。

In [1]:
import torch

# 冒充「被量化的大模型」。重点：它是一个如假包换的 nn.Module
fake_model = torch.nn.Linear(4, 4)

print(type(fake_model))
print("它是 nn.Module 吗？", isinstance(fake_model, torch.nn.Module))

<class 'torch.nn.modules.linear.Linear'>
它是 nn.Module 吗？ True


---
## 情况 A：继承 nn.Module + 有 super().\_\_init\_\_()

这就是**你项目现在的写法**（[quant/quantization/awq/quantizer.py:53](quant/quantization/awq/quantizer.py#L53)）。能跑。

In [9]:
class QuantizerA(torch.nn.Module):   # ← 继承了
    def __init__(self, model):
        super().__init__()           # ← 调了：这一行的作用就是「建账本」
        self.model = model           # ← 看着是普通赋值，其实是「记账」
        self.w_bit = 4               # ← 普通的 int，不记账

a = QuantizerA(fake_model)
print("✅ 能跑")
print(a.modules)

✅ 能跑
<bound method Module.modules of QuantizerA(
  (model): Linear(in_features=4, out_features=4, bias=True)
)>


### 那本「账本」长什么样？

`super().__init__()` 建的就是 `_modules` / `_parameters` / `_buffers` 这几个空字典。
然后 `self.model = model` 那一刻，PyTorch 偷偷把整个模型登记了进去 👇

In [3]:
print("账本 _modules 里有什么：")
print(a._modules)
print()
print("w_bit 在账本里吗？", "w_bit" in a._modules, "  <- int 不是 Module，所以没被记账")
print("model  在账本里吗？", "model"  in a._modules, "  <- 整个大模型被登记成了 quantizer 的『子模块』")

账本 _modules 里有什么：
{'model': Linear(in_features=4, out_features=4, bias=True)}

w_bit 在账本里吗？ False   <- int 不是 Module，所以没被记账
model  在账本里吗？ True   <- 整个大模型被登记成了 quantizer 的『子模块』


### 副作用：quantizer 现在「以为」自己是个神经网络

因为账本记着整个模型，所以 quantizer 凭空长出了一堆本不该有的能力。
这些能力**你的代码一次都没用过**，但它们真实存在：

In [4]:
print("a.parameters() ->", [tuple(p.shape) for p in a.parameters()])
print("a.state_dict() ->", list(a.state_dict().keys()))
print()
print("a.to('cpu') 会搬动整个模型；a.eval() / a.train() 也都能调。")
print("→ 一个『量化脚本』被硬塞了一套神经网络的身份。")

a.parameters() -> [(4, 4), (4,)]
a.state_dict() -> ['model.weight', 'model.bias']

a.to('cpu') 会搬动整个模型；a.eval() / a.train() 也都能调。
→ 一个『量化脚本』被硬塞了一套神经网络的身份。


### 副作用 2：命名空间被占用

这就是 [quantizer.py:76](quant/quantization/awq/quantizer.py#L76) 那句注释在说的事 ——

> 返回值这里不能命名为 self.modules，因为 BaseQuantizer 是一个 torch.nn.Module，它也有 self.modules 成员

所以代码里被迫改名成了 `self.target_modules`。

In [ ]:
print("a.modules 是什么？ ->", a.modules)
print("→ 它是 nn.Module 自带的方法，名字被占了，你没法拿 self.modules 存自己的东西。")

---
## 情况 B：继承 nn.Module + **删掉** super().\_\_init\_\_()

你的直觉是「账本反正没用，那删掉建账本的那行呗」。

**这会直接崩。** 因为「记账」这个动作是被强制执行的 —— PyTorch 不管你需不需要账本，
它看到右边是个 `nn.Module` 就一定要登记，找不到账本就当场报错。

In [8]:
class QuantizerB(torch.nn.Module):   # ← 还是继承了
    def __init__(self, model):
        # super().__init__()         # ← 故意删掉：不建账本
        self.model = model           # ← 但这行还是会去「记账」

b = QuantizerB(fake_model)
print("✅ 能跑")
try:
    b = QuantizerB(fake_model)
    print("✅ 能跑")
except AttributeError as e:
    print("💥 崩了:", e)

AttributeError: cannot assign module before Module.__init__() call

报错信息 `cannot assign module before Module.__init__() call`
翻译过来就是：**「你还没建账本呢，不许给我登记 module」**。

PyTorch 内部的逻辑长这样（简化版）：

```python
def __setattr__(self, name, value):        # 每次 self.x = y 都会跑这个
    if isinstance(value, Module):          # 一看：哦，右边是个 nn.Module
        modules = self.__dict__.get('_modules')
        if modules is None:                # 想拿账本，发现没有
            raise AttributeError("cannot assign module before Module.__init__() call")
        modules[name] = value              # 记账
```

注意它**从来没问过**「你以后会用这个账本吗」。

---
## 情况 C：**不继承** + 没有 super().\_\_init\_\_() ← 推荐

关键：**「不继承」和「不调 super」必须成对出现。**

一旦不继承，`self.model = model` 就退化回一行朴素的 Python 赋值 ——
没有魔法、没有账本、没有记账，自然也不需要 super 了。

In [ ]:
class QuantizerC:                    # ← 什么都不继承，就是个普通类
    def __init__(self, model):
        self.model = model           # ← 真·普通赋值
        self.modules = [1, 2, 3]     # ← 名字也自由了，不用改叫 target_modules

c = QuantizerC(fake_model)
print("✅ 能跑")
print("有账本吗？", hasattr(c, "_modules"))
print("self.modules 现在是我自己的：", c.modules)
print("模型还在，功能一点没少：", c.model)

---
## 三种组合对比

| | 继承 nn.Module？ | 有 super().\_\_init\_\_()？ | 结果 |
|---|---|---|---|
| **A** 现在的代码 | 是 | 是 | ✅ 能跑，但白建了个没人翻的账本 |
| **B** 只删 super | 是 | 否 | 💥 崩 |
| **C** 推荐 | 否 | 否 | ✅ 能跑，且干净 —— 账本根本不存在 |

**为什么 quantizer 属于 C：** 它站在计算图外面，只是「读模型 → 跑校准 → 改权重 → 换 Linear 层」。
它自己不做前向、没有 `forward`、没有 `nn.Parameter`、没有 `register_buffer`。

---
## 对照：什么才**真的**需要继承 nn.Module

同一个文件里的 `Catcher`（[quantizer.py:583](quant/quantization/awq/quantizer.py#L583)）就是正面例子 ——
它要被**塞进模型里**替换掉第 0 层 decoder layer，靠 `forward` 拦截输入来抓 `self.inps`。
它活在计算图里，所以必须是 nn.Module。

In [ ]:
class Catcher(torch.nn.Module):      # ← 这个继承是对的！
    def __init__(self, module):
        super().__init__()
        self.ori_module = module

    def forward(self, x):            # ← 有 forward，会被模型调用
        print(f"   [Catcher] 抓到输入了! shape={tuple(x.shape)}")
        raise ValueError("抓到就够了，中断前向")

# 把模型的某一层替换成 Catcher，然后跑一次前向
net = torch.nn.Sequential(torch.nn.Linear(4, 4), torch.nn.Linear(4, 4))
net[1] = Catcher(net[1])             # ← 偷梁换柱

print("跑前向：")
try:
    net(torch.randn(2, 4))
except ValueError as e:
    print(f"   捕获中断: {e}")

print("\n→ Catcher 参与了前向，所以它必须是 nn.Module。")
print("→ AwqQuantizer 从不参与前向，所以它不必是。")

In [ ]:
def fruit_generator():
    print("开始执行：准备给你第一个水果...")
    yield "🍎 苹果"  # 到这里会交出苹果，然后【暂停】

    print("恢复执行：准备给你第二个水果...")
    yield "🍌 香蕉"  # 到这里交出香蕉，然后再次【暂停】

    print("最后一次恢复：给你最后一个...")
    yield "🍉 西瓜"  # 到这里交出西瓜，然后再次【暂停】

# 1. 先拿到这个生成器（注意：此时函数里面的代码一行都还没执行！）
my_gen = fruit_generator()
print("生成器准备好了，但还没开始出水果。")
print("-" * 30)

# 2. 按第一次 next()
# 它会执行到第一个 yield 并停住
fruit1 = next(my_gen)  
print(f"你拿到了: {fruit1}")
print("-" * 30)

# 3. 按第二次 next()
# 它会从上次停住的地方（第一个 yield 后面）继续往下走！
fruit2 = next(my_gen)  
print(f"你又拿到了: {fruit2}")
print("-" * 30)

# 4. 按第三次 next()
# 它会从上次停住的地方（第一个 yield 后面）继续往下走！
fruit3 = next(my_gen)  
print(f"你又拿到了: {fruit3}")

生成器准备好了，但还没开始出水果。
------------------------------
开始执行：准备给你第一个水果...
你拿到了: 🍎 苹果
------------------------------
恢复执行：准备给你第二个水果...
你又拿到了: 🍌 香蕉
最后一次恢复：给你最后一个...
你又拿到了: 🍉 西瓜
